## Why do we need Vector Stores ? 
Vector stores are useful because they let AI systems find relevant information by meaning, not just by exact keywords

## The Challenges of Managing Vectors
Implementing semantic search introduces three major challenges:
Generation: You must generate embedding vectors for massive datasets (e.g., millions of movies)

Storage: Standard relational databases (like MySQL or Oracle) cannot properly store or calculate similarities between high-dimensional embedding vectors

Semantic Search (Performance): Finding the most similar vector among 1 million vectors using a linear search (O(n)) requires 1 million computations, making the application unbearably slow

## What is a Vector Store?
A Vector Store is a system specifically designed to solve these challenges by storing and retrieving data represented as numerical vectors

The 4 Key Features of Vector Stores:

1. Storage: Retains vectors and their associated metadata (like a movie ID or title). Storage can be in-memory (RAM, lost when closed) for quick lookups, or on-disk (persistent) for durability

2. Similarity Search: Allows you to query a vector and retrieve the stored vectors most similar to it 

3. Indexing: To solve the slow O(n) search problem, vector stores use advanced indexing techniques (like clustering or Approximate Nearest Neighbors) Instead of comparing a query to 1 million vectors, indexing groups vectors into clusters (e.g., 10 clusters of 100,000 vectors). The system first compares the query to the 10 cluster centroids, identifies the closest cluster, and then searches only within those 100,000 vectors, drastically speeding up the process

4. CRUD Operations: Just like a normal database, it supports Create, Retrieve, Update, and Delete operations for your vectors


## Vector Store vs. Vector Database
While often used interchangeably, there is a technical difference
- Vector Store: A lightweight library focused strictly on storing vectors and performing similarity search (e.g., Facebook's FAISS). Ideal for prototyping

- Vector Database: A full-fledged database system. It is essentially a Vector Store bundled with traditional enterprise database features like distributed architecture, ACID transactions, backup/restore, concurrency control, and role-based authentication. Examples include ChromaDB, Pinecone, Milvus, Qdrant, and Weaviate



In [ ]:
# LangChain provides built-in wrappers for all major vector stores
# LangChain intentionally uses standard method signatures (like add_documents or similarity_search) across all its wrappers. 
# This means you can easily swap out Chroma for Pinecone or FAISS without drastically altering your code

from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document

# Create Document Objects (Text + Metadata)
docs = [
    Document(page_content="Virat Kohli is a batsman", metadata={"team": "RCB"}),
    Document(page_content="Rohit Sharma is a batsman", metadata={"team": "MI"}),
    Document(page_content="MS Dhoni is a wicketkeeper", metadata={"team": "CSK"}),
    Document(page_content="Jasprit Bumrah is a bowler", metadata={"team": "MI"}),
    Document(page_content="Ravindra Jadeja is an allrounder", metadata={"team": "CSK"})
]

# You specify the embedding model, a storage directory, and a collection (table) name.
vectorstore = Chroma(
    embedding_function=OpenAIEmbeddings(),
    persist_directory="./my_chroma_db", 
    collection_name="sample"
)

In [ ]:
# B. CRUD Operations & Search Methods
# --- CREATE: Add Documents ---
# Automatically generates embeddings, assigns unique IDs, and stores them in the DB.
vectorstore.add_documents(docs)

# --- RETRIEVE: View All Stored Data ---
# Returns existing IDs, embeddings, documents, and metadatas
db_content = vectorstore.get(include=["embeddings", "documents", "metadatas"])
print(db_content)

# --- SEARCH: Standard Similarity Search ---
# Find the 'k' most semantically similar documents to the query.
results = vectorstore.similarity_search(query="Who among these are a bowler?", k=2)
print(results) 

# --- SEARCH: Similarity Search with Score ---
# Returns the documents along with their similarity distance scores (lower is better/closer).
results_with_score = vectorstore.similarity_search_with_score("Who is a bowler?", k=1)

# --- SEARCH: Metadata Filtering ---
# Search specifically within a filtered subset based on metadata.
filtered_results = vectorstore.similarity_search(
    query="", 
    filter={"team": "CSK"} # Only searches documents tagged with CSK
)

# --- UPDATE: Modify an Existing Document ---
updated_doc = Document(
    page_content="Virat Kohli, the former captain of RCB, is renowned for his leadership.", 
    metadata={"team": "RCB"}
)
doc_id_to_update = db_content['ids'] # Extracting the unique ID

vectorstore.update_document(document_id=doc_id_to_update, document=updated_doc)

# --- DELETE: Remove a Document ---
vectorstore.delete(ids=[doc_id_to_update])